# EDA Silver Amazon 2023 trên Google Colab

Notebook này đọc tầng Silver từ Hugging Face dataset `chuongdo1104/amazon-2023-silver` và EDA từng artifact:

- `silver/silver_item_popularity.parquet/`
- `silver/silver_item_text_profile.parquet/`
- `silver/silver_interactions_train.parquet/`
- `silver/silver_interactions_val.parquet/`
- `silver/silver_interactions_test.parquet/`
- `silver/silver_user_text_profile.parquet/`
- `silver/silver_val_ground_truth.parquet/`
- `silver/silver_test_ground_truth.parquet/`

Mục tiêu: thống kê số lượng data, phân bố user/item/interactions, null, trùng lặp, nhiễu, k-core, Head/Mid/Tail/COLD_START, text profile coverage, ground truth, và các biểu đồ làm cơ sở chọn ngưỡng popularity.

In [ ]:
# Cell 1 - Cài thư viện cho Colab
!pip -q install huggingface_hub duckdb pyarrow pandas numpy matplotlib seaborn tqdm

In [ ]:
# Cell 2 - Imports và cấu hình
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import hf_hub_download, list_repo_files
from IPython.display import display, Markdown
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titleweight'] = 'bold'

REPO_ID = 'chuongdo1104/amazon-2023-silver'
REPO_TYPE = 'dataset'
LOCAL_DIR = Path('/content/amazon_2023_silver')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACTS = {
    'item_popularity': 'silver/silver_item_popularity.parquet',
    'item_text_profile': 'silver/silver_item_text_profile.parquet',
    'interactions_train': 'silver/silver_interactions_train.parquet',
    'interactions_val': 'silver/silver_interactions_val.parquet',
    'interactions_test': 'silver/silver_interactions_test.parquet',
    'user_text_profile': 'silver/silver_user_text_profile.parquet',
    'val_ground_truth': 'silver/silver_val_ground_truth.parquet',
    'test_ground_truth': 'silver/silver_test_ground_truth.parquet',
}

GROUP_ORDER = ['HEAD', 'MID', 'TAIL', 'COLD_START']
display(Markdown(f'**Dataset:** `{REPO_ID}`  \n**Local cache:** `{LOCAL_DIR}`'))

In [ ]:
# Cell 3 - Liệt kê và tải toàn bộ parquet parts của từng Silver folder
repo_files = list_repo_files(REPO_ID, repo_type=REPO_TYPE)
silver_files = sorted([f for f in repo_files if f.startswith('silver/')])
display(pd.DataFrame({'path': silver_files}).head(80))
print(f'Tổng số file trong silver: {len(silver_files):,}')

def parquet_files_under(prefix: str):
    prefix = prefix.rstrip('/') + '/'
    return sorted([f for f in silver_files if f.startswith(prefix) and f.endswith('.parquet')])

downloaded = {}
for artifact, prefix in ARTIFACTS.items():
    files = parquet_files_under(prefix)
    if not files:
        raise FileNotFoundError(f'Không thấy parquet part trong {prefix}')
    local_paths = []
    for f in tqdm(files, desc=f'Download {artifact}'):
        local_paths.append(hf_hub_download(
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            filename=f,
            local_dir=LOCAL_DIR,
            local_dir_use_symlinks=False,
        ))
    downloaded[artifact] = local_paths

download_summary = pd.DataFrame([
    {'artifact': k, 'parquet_files': len(v), 'local_glob': str(LOCAL_DIR / ARTIFACTS[k] / '**/*.parquet')}
    for k, v in downloaded.items()
])
display(download_summary)

In [ ]:
# Cell 4 - Tạo DuckDB views
con = duckdb.connect(database=':memory:')

def sql_path(artifact: str) -> str:
    return str(LOCAL_DIR / ARTIFACTS[artifact] / '**/*.parquet')

for artifact in ARTIFACTS:
    con.execute(f"CREATE OR REPLACE VIEW {artifact} AS SELECT * FROM read_parquet('{sql_path(artifact)}')")

con.execute("""
CREATE OR REPLACE VIEW interactions_all AS
SELECT 'train' AS split, * FROM interactions_train
UNION ALL SELECT 'val' AS split, * FROM interactions_val
UNION ALL SELECT 'test' AS split, * FROM interactions_test
""")
con.execute("CREATE OR REPLACE VIEW user_freq_all AS SELECT reviewer_id, COUNT(*) AS n_interactions FROM interactions_all GROUP BY reviewer_id")
con.execute("CREATE OR REPLACE VIEW item_freq_all AS SELECT parent_asin, COUNT(*) AS n_interactions FROM interactions_all GROUP BY parent_asin")
con.execute("CREATE OR REPLACE VIEW user_freq_train AS SELECT reviewer_id, COUNT(*) AS train_interactions FROM interactions_train GROUP BY reviewer_id")
con.execute("CREATE OR REPLACE VIEW item_freq_train AS SELECT parent_asin, COUNT(*) AS train_interactions FROM interactions_train GROUP BY parent_asin")

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

display(q("""
SELECT 'item_popularity' artifact, COUNT(*) rows FROM item_popularity
UNION ALL SELECT 'item_text_profile', COUNT(*) FROM item_text_profile
UNION ALL SELECT 'interactions_train', COUNT(*) FROM interactions_train
UNION ALL SELECT 'interactions_val', COUNT(*) FROM interactions_val
UNION ALL SELECT 'interactions_test', COUNT(*) FROM interactions_test
UNION ALL SELECT 'user_text_profile', COUNT(*) FROM user_text_profile
UNION ALL SELECT 'val_ground_truth', COUNT(*) FROM val_ground_truth
UNION ALL SELECT 'test_ground_truth', COUNT(*) FROM test_ground_truth
"""))

In [ ]:
# Cell 5 - Schema và sample từng artifact
for name in ARTIFACTS:
    display(Markdown(f'### Schema: `{name}`'))
    display(q(f'DESCRIBE {name}'))
    display(Markdown(f'### Sample rows: `{name}`'))
    display(q(f'SELECT * FROM {name} LIMIT 5'))

In [ ]:
# Cell 6 - Tổng quan interactions train/val/test
split_overview = q("""
SELECT
  split,
  COUNT(*) AS interactions,
  COUNT(DISTINCT reviewer_id) AS users,
  COUNT(DISTINCT parent_asin) AS items,
  ROUND(COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT reviewer_id), 0), 3) AS avg_interactions_per_user,
  ROUND(COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT parent_asin), 0), 3) AS avg_interactions_per_item,
  AVG(rating) AS avg_rating,
  MIN(to_timestamp(timestamp)) AS min_time,
  MAX(to_timestamp(timestamp)) AS max_time
FROM interactions_all
GROUP BY split
ORDER BY CASE split WHEN 'train' THEN 1 WHEN 'val' THEN 2 ELSE 3 END
""")
display(split_overview)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=split_overview, x='split', y='interactions', ax=axes[0])
axes[0].set_title('Interactions theo split')
sns.barplot(data=split_overview, x='split', y='users', ax=axes[1])
axes[1].set_title('Users theo split')
sns.barplot(data=split_overview, x='split', y='items', ax=axes[2])
axes[2].set_title('Items theo split')
for ax in axes:
    ax.ticklabel_format(axis='y', style='plain')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 - Null, duplicate, nhiễu trong từng artifact
def null_report(table: str) -> pd.DataFrame:
    cols = q(f'DESCRIBE {table}')['column_name'].tolist()
    expr = ', '.join([f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in cols])
    df = q(f'SELECT COUNT(*) AS rows, {expr} FROM {table}')
    rows = int(df.loc[0, 'rows'])
    out = df.drop(columns=['rows']).T.reset_index()
    out.columns = ['column', 'null_count']
    out['null_pct'] = 100 * out['null_count'] / max(rows, 1)
    out.insert(0, 'artifact', table)
    return out.sort_values('null_pct', ascending=False)

missing_all = pd.concat([null_report(t) for t in ARTIFACTS], ignore_index=True)
display(missing_all)

quality_inter = q("""
SELECT
  split,
  COUNT(*) AS rows,
  SUM(CASE WHEN reviewer_id IS NULL OR trim(reviewer_id) = '' THEN 1 ELSE 0 END) AS bad_reviewer_id,
  SUM(CASE WHEN parent_asin IS NULL OR trim(parent_asin) = '' THEN 1 ELSE 0 END) AS bad_parent_asin,
  SUM(CASE WHEN timestamp IS NULL OR timestamp <= 0 THEN 1 ELSE 0 END) AS bad_timestamp,
  SUM(CASE WHEN rating IS NULL OR rating < 1 OR rating > 5 THEN 1 ELSE 0 END) AS bad_rating,
  SUM(CASE WHEN train_freq IS NULL OR train_freq < 0 THEN 1 ELSE 0 END) AS bad_train_freq,
  SUM(CASE WHEN popularity_group NOT IN ('HEAD','MID','TAIL','COLD_START') OR popularity_group IS NULL THEN 1 ELSE 0 END) AS bad_popularity_group,
  COUNT(*) - COUNT(DISTINCT reviewer_id || '|' || parent_asin) AS duplicate_user_item_rows
FROM interactions_all
GROUP BY split
ORDER BY split
""")
display(quality_inter)

dup_artifact = q("""
SELECT 'item_popularity' artifact, COUNT(*) - COUNT(DISTINCT parent_asin) duplicate_key_rows FROM item_popularity
UNION ALL SELECT 'item_text_profile', COUNT(*) - COUNT(DISTINCT parent_asin) FROM item_text_profile
UNION ALL SELECT 'user_text_profile', COUNT(*) - COUNT(DISTINCT reviewer_id) FROM user_text_profile
UNION ALL SELECT 'val_ground_truth', COUNT(*) - COUNT(DISTINCT reviewer_id || '|' || parent_asin) FROM val_ground_truth
UNION ALL SELECT 'test_ground_truth', COUNT(*) - COUNT(DISTINCT reviewer_id || '|' || parent_asin) FROM test_ground_truth
""")
display(dup_artifact)

In [ ]:
# Cell 8 - Phân bố popularity group từ item_popularity và item_text_profile
pop_summary = q("""
SELECT popularity_group,
       COUNT(*) AS items,
       SUM(train_freq) AS train_interactions,
       MIN(train_freq) AS min_train_freq,
       quantile_cont(train_freq, 0.50) AS p50_train_freq,
       quantile_cont(train_freq, 0.90) AS p90_train_freq,
       quantile_cont(train_freq, 0.99) AS p99_train_freq,
       MAX(train_freq) AS max_train_freq,
       AVG(train_freq) AS avg_train_freq
FROM item_popularity
GROUP BY popularity_group
ORDER BY CASE popularity_group WHEN 'HEAD' THEN 1 WHEN 'MID' THEN 2 WHEN 'TAIL' THEN 3 ELSE 4 END
""")
pop_summary['item_pct'] = 100 * pop_summary['items'] / pop_summary['items'].sum()
pop_summary['interaction_pct'] = 100 * pop_summary['train_interactions'] / pop_summary['train_interactions'].sum()
display(pop_summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=pop_summary, x='popularity_group', y='items', order=GROUP_ORDER, ax=axes[0])
axes[0].set_title('Số item theo popularity group')
sns.barplot(data=pop_summary, x='popularity_group', y='train_interactions', order=GROUP_ORDER, ax=axes[1])
axes[1].set_title('Train interactions theo popularity group')
freq_plot = q("SELECT popularity_group, train_freq FROM item_popularity")
sns.boxplot(data=freq_plot, x='popularity_group', y='train_freq', order=['HEAD','MID','TAIL'], ax=axes[2])
axes[2].set_yscale('log')
axes[2].set_title('Train_freq theo H/M/T, log y')
plt.tight_layout()
plt.show()

text_group = q("""
SELECT popularity_group, COUNT(*) AS item_text_rows, AVG(token_estimate) AS avg_token_estimate
FROM item_text_profile
GROUP BY popularity_group
ORDER BY CASE popularity_group WHEN 'HEAD' THEN 1 WHEN 'MID' THEN 2 WHEN 'TAIL' THEN 3 ELSE 4 END
""")
display(text_group)

In [ ]:
# Cell 9 - Histogram/CDF để xác định ngưỡng Head/Mid/Tail
freq = q("SELECT parent_asin, train_freq, popularity_group FROM item_popularity ORDER BY train_freq DESC, parent_asin")
display(q("""
SELECT
  MIN(train_freq) min_freq,
  quantile_cont(train_freq, 0.50) p50,
  quantile_cont(train_freq, 0.70) p70,
  quantile_cont(train_freq, 0.80) p80,
  quantile_cont(train_freq, 0.90) p90,
  quantile_cont(train_freq, 0.95) p95,
  quantile_cont(train_freq, 0.99) p99,
  MAX(train_freq) max_freq
FROM item_popularity
"""))

cutoffs = q("""
SELECT popularity_group, MIN(train_freq) min_freq, MAX(train_freq) max_freq
FROM item_popularity
GROUP BY popularity_group
""")
display(cutoffs)
head_cutoff = int(cutoffs.loc[cutoffs['popularity_group'] == 'HEAD', 'min_freq'].iloc[0]) if (cutoffs['popularity_group'] == 'HEAD').any() else None
mid_cutoff = int(cutoffs.loc[cutoffs['popularity_group'] == 'MID', 'min_freq'].iloc[0]) if (cutoffs['popularity_group'] == 'MID').any() else None

freq_sorted = freq.sort_values('train_freq', ascending=False).reset_index(drop=True)
freq_sorted['item_rank_pct'] = 100 * (np.arange(len(freq_sorted)) + 1) / len(freq_sorted)
freq_sorted['cum_interactions_pct'] = 100 * freq_sorted['train_freq'].cumsum() / max(freq_sorted['train_freq'].sum(), 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(freq[freq['train_freq'] > 0]['train_freq'], bins=80, log_scale=(True, True), ax=axes[0])
if head_cutoff is not None:
    axes[0].axvline(head_cutoff, color='red', linestyle='--', label=f'HEAD min={head_cutoff}')
if mid_cutoff is not None:
    axes[0].axvline(mid_cutoff, color='orange', linestyle='--', label=f'MID min={mid_cutoff}')
axes[0].set_title('Histogram train_freq, log-log')
axes[0].set_xlabel('train_freq')
axes[0].legend()

sns.lineplot(data=freq_sorted, x='item_rank_pct', y='cum_interactions_pct', ax=axes[1])
axes[1].axvline(20, color='red', linestyle='--', label='Top 20% items')
axes[1].axvline(30, color='orange', linestyle='--', label='Top 30% items')
axes[1].axhline(80, color='gray', linestyle=':', label='80% interactions')
axes[1].set_title('Cumulative interactions theo item rank')
axes[1].set_xlabel('% item theo rank giảm dần')
axes[1].set_ylabel('% interactions tích lũy')
axes[1].legend()
plt.tight_layout()
plt.show()

display(freq_sorted.head(30))

In [ ]:
# Cell 10 - So sánh các ngưỡng fixed candidate cho Head/Mid/Tail
candidate_rows = []
for tail_max in [1, 2, 3, 5, 10, 20]:
    for head_min in [10, 20, 50, 100]:
        if tail_max >= head_min:
            continue
        tmp = freq.copy()
        tmp['candidate_group'] = np.where(tmp['train_freq'] >= head_min, 'HEAD', np.where(tmp['train_freq'] <= tail_max, 'TAIL', 'MID'))
        g = tmp.groupby('candidate_group').agg(items=('parent_asin', 'count'), interactions=('train_freq', 'sum')).reset_index()
        row = {'rule': f'TAIL<= {tail_max}, HEAD>= {head_min}'}
        for _, r in g.iterrows():
            row[f"{r['candidate_group']}_items_pct"] = 100 * r['items'] / len(tmp)
            row[f"{r['candidate_group']}_interactions_pct"] = 100 * r['interactions'] / max(tmp['train_freq'].sum(), 1)
        candidate_rows.append(row)
candidates = pd.DataFrame(candidate_rows).fillna(0)
display(candidates)

display(Markdown('Bảng này giúp chọn ngưỡng dễ giải thích hơn Pareto. Hãy ưu tiên rule mà TAIL vẫn còn đủ interactions để đánh giá long-tail, còn HEAD không nuốt quá nhiều item.'))

In [ ]:
# Cell 11 - Phân bố interactions theo popularity_group trong train/val/test
group_split = q("""
SELECT split, popularity_group,
       COUNT(*) AS interactions,
       COUNT(DISTINCT reviewer_id) AS users,
       COUNT(DISTINCT parent_asin) AS items,
       AVG(train_freq) AS avg_train_freq
FROM interactions_all
GROUP BY split, popularity_group
ORDER BY split, CASE popularity_group WHEN 'HEAD' THEN 1 WHEN 'MID' THEN 2 WHEN 'TAIL' THEN 3 ELSE 4 END
""")
group_split['interaction_pct_in_split'] = group_split.groupby('split')['interactions'].transform(lambda s: 100 * s / s.sum())
display(group_split)

plt.figure(figsize=(12, 5))
sns.barplot(data=group_split, x='split', y='interaction_pct_in_split', hue='popularity_group', hue_order=GROUP_ORDER)
plt.title('% interactions theo popularity_group trong từng split')
plt.ylabel('% interactions')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 - Ground truth val/test: tail, cold-start, consistency
gt = q("""
SELECT 'val' AS split, * FROM val_ground_truth
UNION ALL SELECT 'test' AS split, * FROM test_ground_truth
""")
display(gt.head())

gt_summary = q("""
WITH gt AS (
  SELECT 'val' AS split, * FROM val_ground_truth
  UNION ALL SELECT 'test' AS split, * FROM test_ground_truth
)
SELECT split, popularity_group,
       COUNT(*) AS rows,
       COUNT(DISTINCT reviewer_id) AS users,
       COUNT(DISTINCT parent_asin) AS items,
       SUM(is_tail) AS tail_rows,
       SUM(is_cold_start) AS cold_start_rows,
       SUM(CASE WHEN is_tail = 1 AND popularity_group <> 'TAIL' THEN 1 ELSE 0 END) AS bad_tail_flag,
       SUM(CASE WHEN is_cold_start = 1 AND popularity_group <> 'COLD_START' THEN 1 ELSE 0 END) AS bad_cold_flag
FROM gt
GROUP BY split, popularity_group
ORDER BY split, CASE popularity_group WHEN 'HEAD' THEN 1 WHEN 'MID' THEN 2 WHEN 'TAIL' THEN 3 ELSE 4 END
""")
gt_summary['row_pct_in_split'] = gt_summary.groupby('split')['rows'].transform(lambda s: 100 * s / s.sum())
display(gt_summary)

plt.figure(figsize=(11, 5))
sns.barplot(data=gt_summary, x='split', y='row_pct_in_split', hue='popularity_group', hue_order=GROUP_ORDER)
plt.title('Ground truth: % row theo popularity_group')
plt.ylabel('% rows')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 - % user theo số tương tác và thống kê user frequency
user_pct = q("""
WITH f AS (
  SELECT split, reviewer_id, COUNT(*) AS n_interactions
  FROM interactions_all
  GROUP BY split, reviewer_id
), total AS (
  SELECT split, COUNT(*) AS n_users FROM f GROUP BY split
)
SELECT f.split, f.n_interactions, COUNT(*) AS users,
       ROUND(100.0 * COUNT(*) / MAX(total.n_users), 4) AS user_pct
FROM f JOIN total USING(split)
GROUP BY f.split, f.n_interactions
ORDER BY f.split, f.n_interactions
""")
display(user_pct[user_pct['n_interactions'] <= 30])

plt.figure(figsize=(14, 5))
sns.lineplot(data=user_pct[user_pct['n_interactions'] <= 30], x='n_interactions', y='user_pct', hue='split', marker='o')
plt.title('% user theo số tương tác trong từng split, cắt tại 30')
plt.xlabel('Số tương tác/user')
plt.ylabel('% user')
plt.tight_layout()
plt.show()

display(q("""
SELECT 'all_splits' AS scope, MIN(n_interactions) min_n,
       quantile_cont(n_interactions, 0.5) p50,
       quantile_cont(n_interactions, 0.9) p90,
       quantile_cont(n_interactions, 0.95) p95,
       quantile_cont(n_interactions, 0.99) p99,
       MAX(n_interactions) max_n, AVG(n_interactions) avg_n
FROM user_freq_all
UNION ALL
SELECT 'train_only', MIN(train_interactions), quantile_cont(train_interactions, 0.5), quantile_cont(train_interactions, 0.9),
       quantile_cont(train_interactions, 0.95), quantile_cont(train_interactions, 0.99), MAX(train_interactions), AVG(train_interactions)
FROM user_freq_train
"""))

In [ ]:
# Cell 14 - % item theo số tương tác và số lượng item có tương tác
item_pct = q("""
WITH f AS (
  SELECT split, parent_asin, COUNT(*) AS n_interactions
  FROM interactions_all
  GROUP BY split, parent_asin
), total AS (
  SELECT split, COUNT(*) AS n_items FROM f GROUP BY split
)
SELECT f.split, f.n_interactions, COUNT(*) AS items,
       ROUND(100.0 * COUNT(*) / MAX(total.n_items), 4) AS item_pct
FROM f JOIN total USING(split)
GROUP BY f.split, f.n_interactions
ORDER BY f.split, f.n_interactions
""")
display(item_pct[item_pct['n_interactions'] <= 30])

plt.figure(figsize=(14, 5))
sns.lineplot(data=item_pct[item_pct['n_interactions'] <= 30], x='n_interactions', y='item_pct', hue='split', marker='o')
plt.title('% item theo số tương tác trong từng split, cắt tại 30')
plt.xlabel('Số tương tác/item')
plt.ylabel('% item')
plt.tight_layout()
plt.show()

display(q("""
SELECT 'all_splits' AS scope, MIN(n_interactions) min_n,
       quantile_cont(n_interactions, 0.5) p50,
       quantile_cont(n_interactions, 0.9) p90,
       quantile_cont(n_interactions, 0.95) p95,
       quantile_cont(n_interactions, 0.99) p99,
       MAX(n_interactions) max_n, AVG(n_interactions) avg_n
FROM item_freq_all
UNION ALL
SELECT 'train_only', MIN(train_interactions), quantile_cont(train_interactions, 0.5), quantile_cont(train_interactions, 0.9),
       quantile_cont(train_interactions, 0.95), quantile_cont(train_interactions, 0.99), MAX(train_interactions), AVG(train_interactions)
FROM item_freq_train
"""))

In [ ]:
# Cell 15 - K-core candidate analysis cho user và item
thresholds = [1, 2, 3, 5, 10, 20, 50, 100]
rows = []
for scope, user_table, item_table, user_col, item_col, source_view in [
    ('all_splits', 'user_freq_all', 'item_freq_all', 'n_interactions', 'n_interactions', 'interactions_all'),
    ('train_only', 'user_freq_train', 'item_freq_train', 'train_interactions', 'train_interactions', 'interactions_train'),
]:
    total_interactions = q(f'SELECT COUNT(*) AS n FROM {source_view}').iloc[0, 0]
    for k in thresholds:
        users = q(f'SELECT COUNT(*) AS n FROM {user_table} WHERE {user_col} >= {k}').iloc[0, 0]
        items = q(f'SELECT COUNT(*) AS n FROM {item_table} WHERE {item_col} >= {k}').iloc[0, 0]
        inter_user = q(f"""
            SELECT COUNT(*) AS n FROM {source_view} i
            JOIN {user_table} u USING(reviewer_id)
            WHERE u.{user_col} >= {k}
        """).iloc[0, 0]
        inter_item = q(f"""
            SELECT COUNT(*) AS n FROM {source_view} i
            JOIN {item_table} it USING(parent_asin)
            WHERE it.{item_col} >= {k}
        """).iloc[0, 0]
        rows.append({
            'scope': scope,
            'k': k,
            'users_ge_k': users,
            'items_ge_k': items,
            'interactions_kept_by_user_k': inter_user,
            'interactions_kept_by_item_k': inter_item,
            'user_k_interaction_pct': 100 * inter_user / max(total_interactions, 1),
            'item_k_interaction_pct': 100 * inter_item / max(total_interactions, 1),
        })
kcore = pd.DataFrame(rows)
display(kcore)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=kcore, x='k', y='users_ge_k', hue='scope', marker='o', ax=axes[0])
axes[0].set_title('Users có interactions >= k')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
sns.lineplot(data=kcore, x='k', y='items_ge_k', hue='scope', marker='o', ax=axes[1])
axes[1].set_title('Items có interactions >= k')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16 - Item text profile EDA: coverage, token, review snippets, source level
item_text_summary = q("""
SELECT popularity_group,
       COUNT(*) AS rows,
       SUM(CASE WHEN item_text IS NULL OR trim(item_text) = '' OR item_text LIKE '[NO_TEXT]%' THEN 1 ELSE 0 END) AS no_text_rows,
       AVG(token_estimate) AS avg_token_estimate,
       quantile_cont(token_estimate, 0.50) AS token_p50,
       quantile_cont(token_estimate, 0.90) AS token_p90,
       quantile_cont(token_estimate, 0.99) AS token_p99,
       AVG(item_review_pos_count) AS avg_pos_review_count,
       AVG(item_review_neg_count) AS avg_neg_review_count
FROM item_text_profile
GROUP BY popularity_group
ORDER BY CASE popularity_group WHEN 'HEAD' THEN 1 WHEN 'MID' THEN 2 WHEN 'TAIL' THEN 3 ELSE 4 END
""")
item_text_summary['no_text_pct'] = 100 * item_text_summary['no_text_rows'] / item_text_summary['rows']
display(item_text_summary)

source_level = q("""
SELECT popularity_group, text_source_level, COUNT(*) AS items
FROM item_text_profile
GROUP BY popularity_group, text_source_level
ORDER BY popularity_group, text_source_level
""")
display(source_level)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(data=q('SELECT popularity_group, token_estimate FROM item_text_profile WHERE token_estimate IS NOT NULL'), x='token_estimate', hue='popularity_group', bins=80, log_scale=(True, True), ax=axes[0])
axes[0].set_title('Token estimate của item_text, log-log')
sns.barplot(data=source_level, x='text_source_level', y='items', hue='popularity_group', hue_order=GROUP_ORDER, ax=axes[1])
axes[1].set_title('Text source level theo group')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 17 - User text profile EDA
user_text_summary = q("""
SELECT
  COUNT(*) AS users,
  SUM(CASE WHEN user_text IS NULL OR trim(user_text) = '' OR user_text LIKE '[NO_TEXT]%' THEN 1 ELSE 0 END) AS no_text_users,
  AVG(review_count_train) AS avg_review_count_train,
  quantile_cont(review_count_train, 0.50) AS review_count_p50,
  quantile_cont(review_count_train, 0.90) AS review_count_p90,
  quantile_cont(review_count_train, 0.99) AS review_count_p99,
  AVG(pos_review_count_train) AS avg_pos_review_count_train,
  AVG(neg_review_count_train) AS avg_neg_review_count_train,
  AVG(avg_rating) AS avg_user_avg_rating,
  AVG(length(coalesce(user_text, ''))) AS avg_user_text_len
FROM user_text_profile
""")
user_text_summary['no_text_user_pct'] = 100 * user_text_summary['no_text_users'] / user_text_summary['users']
display(user_text_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(data=q('SELECT review_count_train FROM user_text_profile'), x='review_count_train', bins=80, log_scale=(True, True), ax=axes[0])
axes[0].set_title('User profile review_count_train, log-log')
sns.histplot(data=q("SELECT length(coalesce(user_text, '')) AS user_text_len FROM user_text_profile"), x='user_text_len', bins=80, log_scale=(True, True), ax=axes[1])
axes[1].set_title('Độ dài user_text, log-log')
plt.tight_layout()
plt.show()

display(q("""
SELECT reviewer_id, review_count_train, pos_review_count_train, neg_review_count_train, avg_rating, avg_review_weight, user_text
FROM user_text_profile
ORDER BY review_count_train DESC
LIMIT 20
"""))

In [ ]:
# Cell 18 - Rating, helpful_vote, temporal distribution
rating_dist = q("""
SELECT split, rating, COUNT(*) AS interactions
FROM interactions_all
GROUP BY split, rating
ORDER BY split, rating
""")
display(rating_dist)

monthly = q("""
SELECT split, year_month, COUNT(*) AS interactions
FROM interactions_all
GROUP BY split, year_month
ORDER BY year_month, split
""")
display(monthly.head(20))
display(monthly.tail(20))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=rating_dist, x='rating', y='interactions', hue='split', ax=axes[0])
axes[0].set_title('Rating distribution')
sns.lineplot(data=monthly, x='year_month', y='interactions', hue='split', marker='o', ax=axes[1])
axes[1].set_title('Interactions theo year_month')
axes[1].tick_params(axis='x', rotation=75)
plt.tight_layout()
plt.show()

display(q("""
SELECT split,
       MIN(helpful_vote) min_helpful,
       quantile_cont(helpful_vote, 0.50) p50,
       quantile_cont(helpful_vote, 0.90) p90,
       quantile_cont(helpful_vote, 0.99) p99,
       MAX(helpful_vote) max_helpful
FROM interactions_all
GROUP BY split
"""))

In [ ]:
# Cell 19 - Top users/items và kiểm tra join giữa artifacts
display(Markdown('### Top users trong interactions_all'))
display(q("""
SELECT reviewer_id, COUNT(*) AS interactions,
       SUM(CASE WHEN split='train' THEN 1 ELSE 0 END) AS train_interactions,
       SUM(CASE WHEN split='val' THEN 1 ELSE 0 END) AS val_interactions,
       SUM(CASE WHEN split='test' THEN 1 ELSE 0 END) AS test_interactions
FROM interactions_all
GROUP BY reviewer_id
ORDER BY interactions DESC
LIMIT 30
"""))

display(Markdown('### Top items theo train_freq kèm text metadata'))
display(q("""
SELECT p.parent_asin, p.train_freq, p.popularity_group, t.title, t.main_category, t.text_source_level, t.token_estimate,
       t.item_review_pos_count, t.item_review_neg_count
FROM item_popularity p
LEFT JOIN item_text_profile t USING(parent_asin)
ORDER BY p.train_freq DESC
LIMIT 30
"""))

join_checks = q("""
WITH inter_items AS (SELECT DISTINCT parent_asin FROM interactions_all),
inter_users AS (SELECT DISTINCT reviewer_id FROM interactions_all)
SELECT 'interaction_items_missing_popularity' AS check_name, COUNT(*) AS rows
FROM inter_items i LEFT JOIN item_popularity p USING(parent_asin) WHERE p.parent_asin IS NULL
UNION ALL
SELECT 'interaction_items_missing_text_profile', COUNT(*)
FROM inter_items i LEFT JOIN item_text_profile t USING(parent_asin) WHERE t.parent_asin IS NULL
UNION ALL
SELECT 'interaction_users_missing_user_profile', COUNT(*)
FROM inter_users u LEFT JOIN user_text_profile p USING(reviewer_id) WHERE p.reviewer_id IS NULL
""")
display(join_checks)

In [ ]:
# Cell 20 - Kết luận nhanh tự động từ các bảng chính
display(Markdown(f"""
### Gợi ý đọc kết quả

- Ngưỡng Silver hiện tại có thể đọc từ Cell 9: `HEAD min`, `MID min`, và `TAIL` nằm dưới `MID min` với `train_freq > 0`.
- Cell 8 cho biết mỗi group chiếm bao nhiêu % item và bao nhiêu % train interactions. Đây là bảng quan trọng nhất để xem long-tail có đủ signal không.
- Cell 11 và Cell 12 cho biết validation/test đang đánh giá nhiều vào HEAD, MID, TAIL hay COLD_START.
- Cell 13-15 dùng để quyết định có cần thêm user/item k-core hoặc lọc dữ liệu trước training không.
- Cell 16-17 kiểm tra chất lượng text profile của Silver, đặc biệt `NO_TEXT`, `token_estimate`, `text_source_level`, positive/negative review snippets.
"""))